# Lesson 08 — 把传感器细节留在 Driver 层

这是一段真实的 C++ 实验：你修改代码格，再运行，就会看到下方深色虚拟设备的输入、输出和检查结果变化。Python 只负责让 VS Code Notebook 启动 C++ 编译和实验台。

## 先运行

按顺序运行初始化格和后面的代码格。绿色 ✓ 表示本格 C++ 符合当前契约；红色 ✗ 会给出预期与实际输出。


In [ ]:
# 运行支架：只运行一次，不需要学习或修改 Python。
from pathlib import Path
project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "tools/notebook_setup.py").is_file()), None)
if project is None:
    raise RuntimeError("请用 VS Code 打开 esp32-hands-on-academy 文件夹")
get_ipython().run_line_magic("run", str(project / "tools/notebook_setup.py"))


上层提醒器只需要一个可靠温度值，却不应到处处理连接、故障和校准细节。

## 最小知识

硬件抽象层（HAL）把设备输入转换为语义输出。`reading` 是校准后的摄氏温度；断连和故障用 `error` 表达，而不是继续使用旧数据。


## 1. 第一个实验（Guided）

先运行。不要急着读每一行；先操作实验台，观察输入与输出之间的关系。


In [ ]:
%%academy_lab 08-driver-hal guided guided pass
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& b) : b_(b) {
        }
        void tick() {
            b_.output.reading = b_.input.temperature + .5;
            b_.output.led = b_.output.reading >= 28.;
        }

    private:
        Device& b_;
    };
}


## 2. 修改一次

把校准偏移从 `0.5` 改成 `-1.0`，确认界面显示的是应用层读数。

重新运行这一个格，再在实验台确认你的预测。


In [ ]:
%%academy_lab 08-driver-hal modify
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& b) : b_(b) {
        }
        void tick() {
            b_.output.reading = b_.input.temperature + .5;
            b_.output.led = b_.output.reading >= 28.;
        }

    private:
        Device& b_;
    };
}


## 3. 故障实验（Debug）

下面代码总是相信温度字段，断线后仍显示一个看似可信的数。运行后失败是预期现象。用面板的时间线、波形、事务或终端找出失败发生在何时。


In [ ]:
%%academy_lab 08-driver-hal broken exercise fail
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& b) : b_(b) {
        }
        void tick() {
            b_.output.reading = b_.input.temperature + .5;
            b_.output.led = b_.output.reading >= 28.;
        }

    private:
        Device& b_;
    };
}


## 4. 你来完成（Build）

实现一个私有读取步骤：只有 connected 且没有 fault 时更新 reading，否则置 error、熄灯并标记 sensor error。

默认这格会失败，失败信息是需求的一部分。先自己尝试；需要时再看 [分层 Hint](../lessons/08-driver-hal/hints.md)。


In [ ]:
%%academy_lab 08-driver-hal exercise exercise fail
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& b) : b_(b) {
        }
        void tick() { // TODO: connected / fault / 校准
        }

    private:
        Device& b_;
    };
}


## 5. 开放挑战

把读取步骤提取为可注入的 `TemperatureDriver` 接口，并用 fake driver 写单元测试。

## 你刚刚真正学到了什么？

Driver/HAL 的价值不是多写一层代码，而是隔离变化：上层根据语义值和错误状态工作。

## 硬件迁移

STEP 2 将私有读取步骤替换成实际 I2C/ADC Driver。校准常数、采样间隔和故障诊断仍由这个边界管理。

完成自己的尝试后才查看 [独立 Solution Notebook](solutions/08-driver-hal-solution.ipynb)。
